# GHIST+ BreastCancer2 Inference

This notebook runs GHIST+ BreastCancer2 fullslide inference using only the H&E image, nuclei segmentation, saved checkpoint, and a compact train-derived reference.


In [2]:
from pathlib import Path
import importlib.util
import os
import shutil
import sys

import numpy as np
import pandas as pd
import torch

REPO = Path(os.environ.get("REPO_DIR", Path.cwd())).expanduser().resolve()
if not (REPO / "tools" / "inference.py").exists():
    raise FileNotFoundError("Run this notebook from the repository root or set REPO_DIR.")
BUNDLE = Path(os.environ.get("EXAMPLE_BUNDLE", REPO.parent / "example_bundle")).expanduser().resolve()

CONFIG = REPO / "configs" / "config_breast2_inference_example.json"
INFERENCE_PY = REPO / "tools" / "inference.py"
ARTIFACTS = BUNDLE / "model_artifacts" / "breast"
CHECKPOINT = ARTIFACTS / "best_model" / "best_model.pth"
IMPUTE_DIR = ARTIFACTS / "imputed_33f50e3f_fixedsvg10"
AVGEXP_REF = ARTIFACTS / "avgexp_ref_slide_specific_train_derived.npz"
CACHE_DIR = ARTIFACTS / "cache"
OUTPUT_DIR = BUNDLE / "inference_outputs" / "breast2_fullslide"

os.environ["DATA_ROOT"] = str(BUNDLE / "example_data")
os.environ["RUN_ROOT"] = str(BUNDLE / "model_artifacts")
os.environ["CACHE_ROOT"] = str(ARTIFACTS)

required = [
    CONFIG,
    INFERENCE_PY,
    CHECKPOINT,
    ARTIFACTS / "genes.txt",
    ARTIFACTS / "standardisation_hist_fold_1.npy",
    AVGEXP_REF,
    IMPUTE_DIR / "test_slide3_domain0_mask.npy",
    BUNDLE / "example_data" / "breast" / "aligned_he_image.tif",
    BUNDLE / "example_data" / "data_processing" / "data_processing_breast2" / "he_image.tif",
    BUNDLE / "example_data" / "data_processing" / "data_processing_breast2" / "he_image_nuclei_seg.tif",
    BUNDLE / "example_data" / "data_processing" / "data_processing_breast2" / "matched_nuclei_filtered.csv",
    BUNDLE / "example_data" / "data_processing" / "data_processing_breast2" / "cell_coords.csv",
]
cache_files = sorted(CACHE_DIR.glob("dataset_*.pt"))
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required files:\n" + "\n".join(missing))
if not cache_files:
    raise FileNotFoundError(f"Missing stripped no-target cache in {CACHE_DIR}")

print("Repo:", REPO)
print("Bundle:", BUNDLE)
print("Output:", OUTPUT_DIR)
print("Bundled cache:", cache_files[0])


Repo: /data1/gzha9095/GHIST_plus
Bundle: /data1/gzha9095/example_bundle
Output: /data1/gzha9095/example_bundle/inference_outputs/breast2_fullslide
Bundled cache: /data1/gzha9095/example_bundle/model_artifacts/breast/cache/dataset_12458a419bf45132fc16465b13cb429c.pt


In [3]:
def load_module(path, name):
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def ensure_no_target_cache(trainer):
    from dataio.dataset_input import _build_cache_path
    from utils.utils import json_file_to_pyobj, read_txt

    opts = trainer._to_namespace(json_file_to_pyobj(str(CONFIG)))
    source = next(s for s in opts.data_sources_test if int(s.slide_idx) == 3)
    gene_names = read_txt(str(ARTIFACTS / "genes.txt"))
    fold_id = 1
    divisions_fold = opts.regions_test.divisions[fold_id - 1]
    meta = {
        "fp_hist": source.fp_hist,
        "fp_nuc_seg": source.fp_nuc_seg,
        "fp_expr": None,
        "fp_cell_type": None,
        "fp_nuc_sizes": source.fp_nuc_sizes,
        "mode": "test",
        "fold_id": fold_id,
        "hsize": int(opts.data.hsize),
        "wsize": int(opts.data.wsize),
        "overlap": int(opts.data.overlap),
        "divisions": divisions_fold,
        "gene_names": gene_names,
        "hist_mtime": os.path.getmtime(source.fp_hist),
        "nuc_mtime": os.path.getmtime(source.fp_nuc_seg),
        "expr_mtime": None,
        "celltype_mtime": None,
        "nuc_sizes_mtime": os.path.getmtime(source.fp_nuc_sizes),
    }
    expected_cache = _build_cache_path(str(ARTIFACTS), meta)
    bundled_cache = sorted(CACHE_DIR.glob("dataset_*.pt"))[0]

    payload = torch.load(bundled_cache, map_location="cpu", weights_only=False)
    if payload.get("df_expr") is not None or payload.get("df_ct") is not None:
        raise RuntimeError("Bundled cache is not target-free: df_expr/df_ct must be None.")
    if expected_cache != bundled_cache and not expected_cache.exists():
        shutil.copy2(bundled_cache, expected_cache)
    print("No-target cache:", expected_cache)
    return expected_cache


def use_precomputed_avgexp(trainer, ref_path):
    ref = np.load(ref_path, allow_pickle=True)
    ref_slide_ids = [int(x) for x in ref["slide_ids"].tolist()]
    ref_by_slide = ref["expr_ref_by_slide"].astype(np.float32)
    ref_mean = ref["expr_ref_mean"].astype(np.float32)
    ref_genes = [str(x) for x in ref["gene_names"].tolist()]
    ref_classes = [str(x) for x in ref["classes"].tolist()]

    def build_avgexp_df_by_slide(all_sources, stats_sources, gene_names, classes, expr_scale, **kwargs):
        gene_pos = {g: i for i, g in enumerate(ref_genes)}
        class_pos = {c: i for i, c in enumerate(ref_classes)}
        missing_genes = [g for g in gene_names if g not in gene_pos]
        missing_classes = [c for c in classes if c not in class_pos]
        if missing_genes or missing_classes:
            raise ValueError(
                f"Avgexp reference mismatch: missing_genes={len(missing_genes)} "
                f"missing_classes={len(missing_classes)}"
            )

        gene_idx = [gene_pos[g] for g in gene_names]
        class_idx = [class_pos[c] for c in classes]
        mean_df = pd.DataFrame(ref_mean[np.ix_(class_idx, gene_idx)], index=classes, columns=gene_names)
        slide_df = {
            sid: pd.DataFrame(ref_by_slide[i][np.ix_(class_idx, gene_idx)], index=classes, columns=gene_names)
            for i, sid in enumerate(ref_slide_ids)
        }
        return {
            int(getattr(src, "slide_idx", -1)): slide_df.get(int(getattr(src, "slide_idx", -1)), mean_df)
            for src in all_sources
        }

    trainer.reference_utils.build_avgexp_df_by_slide = build_avgexp_df_by_slide
    trainer.build_avgexp_df_by_slide = build_avgexp_df_by_slide
    return trainer


inference = load_module(INFERENCE_PY, "inference_module")
trainer = inference._load_nature_trainer(REPO)
trainer = use_precomputed_avgexp(trainer, AVGEXP_REF)
ensure_no_target_cache(trainer)
inference._load_nature_trainer = lambda nature_root: trainer

print("Using avgexp reference:", AVGEXP_REF)


No-target cache: /data1/gzha9095/example_bundle/model_artifacts/breast/cache/dataset_12458a419bf45132fc16465b13cb429c.pt
Using avgexp reference: /data1/gzha9095/example_bundle/model_artifacts/breast/avgexp_ref_slide_specific_train_derived.npz


In [4]:
GPU_ID = 0
BATCH_SIZE = 64
NUM_WORKERS = 32
LOG_EVERY = 10

argv = [
    "tools/inference.py",
    "--nature_root", str(REPO),
    "--experiment_path", str(ARTIFACTS),
    "--config_file", str(CONFIG),
    "--checkpoint_path", str(CHECKPOINT),
    "--impute_dir", str(IMPUTE_DIR),
    "--fold_id", "1",
    "--slide_id", "3",
    "--gpu_id", str(GPU_ID),
    "--num_workers", str(NUM_WORKERS),
    "--batch_size", str(BATCH_SIZE),
    "--output_dir", str(OUTPUT_DIR),
    "--log_every", str(LOG_EVERY),
    "--skip_metrics",
]

old_argv = sys.argv
try:
    sys.argv = argv
    inference.main()
finally:
    sys.argv = old_argv


[INFO] CACHE_ROOT=/data1/gzha9095/example_bundle/model_artifacts/breast
[INFO] checkpoint candidates (best-first): ['best_model.pth']
Using GPUs: 0
[INFO] device=cuda
[INFO] n_genes=413 n_classes=9
[INFO] eval_slide=3 split=test domain_id=0
[INFO] out_dir=/data1/gzha9095/example_bundle/inference_outputs/breast2_fullslide
[INFO] targets unavailable; export dataset will use mode=test
Do stain normalisation: 1
Raising max_cells_per_patch from 600 to 2048 for dense-slide stability.
[cache] Loaded dataset metadata from /data1/gzha9095/example_bundle/model_artifacts/breast/cache/dataset_12458a419bf45132fc16465b13cb429c.pt
[INFO] dataloader batches=243 batch_size=64
[INFO] avgexp ref mode=global trainval-only
[INFO] slide-specific ref map built from trainval stats for slides: [0, 1, 2, 3, 4, 5]
[INFO] using slide-specific train-derived ref for eval slide 3
[INFO] backbone pretrained loading disabled for inference; checkpoint weights will be used
[INFO] checkpoint selected=/data1/gzha9095/exam

RuntimeError: Checkpoint/model mismatch: missing=0 unexpected=2